# 4DFlowNet Mini: Colab Experiment

This notebook recreates the core 4DFlowNet idea on cheap synthetic data:

1. Generate high-resolution 3D velocity fields.
2. Downsample and corrupt them to imitate low-resolution 4D flow MRI.
3. Train a small 4DFlowNet-style residual CNN.
4. Compare against trilinear interpolation using velocity and flow metrics.

This is a controlled reproduction of the mechanism, not a full CFD/aortic clinical reproduction.

## 0. Colab Setup

In Colab, set **Runtime -> Change runtime type -> GPU** first. Then either upload the `4dflownet-mini` folder or clone your repo and `cd` into it.

In [ ]:
!git clone https://github.com/eliasubz/4D-FlowNet
%cd 4D-FlowNet
!pip -q install -r requirements.txt

Cloning into '4D-FlowNet'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 26 (delta 1), reused 26 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 328.16 KiB | 41.02 MiB/s, done.
Resolving deltas: 100% (1/1), done.
[Errno 2] No such file or directory: 'eliasubz/4D-FlowNet'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [1]:
# If you uploaded this folder manually, adjust this path if needed.
# Example for Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/4dflownet-mini

import os
from pathlib import Path

if Path('/content/4dflownet-mini').exists():
    os.chdir('/content/4dflownet-mini')

print('cwd =', os.getcwd())

cwd = /content


In [2]:
!pip -q install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 1. Imports And Config

In [3]:
import math
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.dataset import SyntheticFlowDataset, upsample_lr
from src.losses import four_d_flow_loss
from src.metrics import divergence_l1, endpoint_error, flow_rate_error, peak_velocity_error
from src.model import FourDFlowNet
from src.synthetic_flows import make_grid
from src.visualize import make_interactive_3d_flow_figure

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

seed = 7
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(seed)

ModuleNotFoundError: No module named 'src'

In [ ]:
# Small defaults are intentional. Increase after the first successful run.
config = {
    'hr_size': 32,
    'scale': 2,
    'noise_std': 0.03,
    'train_samples': 512,
    'val_samples': 96,
    'batch_size': 4,
    'epochs': 10,
    'lr': 1e-3,
    'width': 24,
    'lr_blocks': 4,
    'hr_blocks': 2,
    'gradient_weight': 1e-3,
    'divergence_weight': 0.0,
}
config

## 2. Preview One Synthetic Datapoint

In [ ]:
def speed(v):
    return torch.linalg.vector_norm(v, dim=0)

preview_dataset = SyntheticFlowDataset(samples=1, hr_size=config['hr_size'], noise_std=config['noise_std'], seed=42)
lr_input, hr = preview_dataset[0]
lr_velocity = lr_input[:3].unsqueeze(0)
interp = upsample_lr(lr_velocity, hr.shape[-1]).squeeze(0)

hr_speed = speed(hr)
interp_speed = speed(interp)
error = speed(interp - hr)
z = hr.shape[1] // 2
y = hr.shape[2] // 2

fig = plt.figure(figsize=(13, 7), constrained_layout=True)
grid = fig.add_gridspec(2, 3)
axes = {
    'hr_xy': fig.add_subplot(grid[0, 0]),
    'field3d': fig.add_subplot(grid[0, 1], projection='3d'),
    'err_xy': fig.add_subplot(grid[0, 2]),
    'hr_xz': fig.add_subplot(grid[1, 0]),
    'quiver': fig.add_subplot(grid[1, 1]),
    'hist': fig.add_subplot(grid[1, 2]),
}

vmax = hr_speed.max().item()
for key, title, image, cmap, vmin, vmax_i in [
    ('hr_xy', 'HR speed, axial slice', hr_speed[z], 'viridis', 0.0, vmax),
    ('err_xy', 'Interpolation error', error[z], 'magma', 0.0, error.max().item()),
    ('hr_xz', 'HR speed, sagittal slice', hr_speed[:, y, :], 'viridis', 0.0, vmax),
]:
    im = axes[key].imshow(image.cpu(), origin='lower', cmap=cmap, vmin=vmin, vmax=vmax_i)
    axes[key].set_title(title)
    axes[key].set_xticks([])
    axes[key].set_yticks([])
    fig.colorbar(im, ax=axes[key], fraction=0.046, pad=0.04)

x, yy, zz = make_grid(hr.shape[-1])
mask = hr_speed > 0.12
sampled = torch.zeros_like(mask, dtype=torch.bool)
sampled[::4, ::4, ::4] = True
mask = mask & sampled
color3 = hr_speed[mask].cpu()
axes['field3d'].quiver(
    x[mask].cpu(), yy[mask].cpu(), zz[mask].cpu(),
    hr[0][mask].cpu(), hr[1][mask].cpu(), hr[2][mask].cpu(),
    length=0.16, normalize=True,
    colors=plt.cm.viridis(color3 / color3.max()),
)
axes['field3d'].set_title('3D velocity field')
axes['field3d'].set_xlim(-1, 1); axes['field3d'].set_ylim(-1, 1); axes['field3d'].set_zlim(-1, 1)
axes['field3d'].set_xticks([]); axes['field3d'].set_yticks([]); axes['field3d'].set_zticks([])
axes['field3d'].view_init(elev=22, azim=-58)

step = 2
axes['quiver'].imshow(hr_speed[z].cpu(), origin='lower', cmap='Greys', alpha=0.45)
axes['quiver'].quiver(hr[0, z, ::step, ::step].cpu(), hr[1, z, ::step, ::step].cpu(), color='tab:red', angles='xy', scale_units='xy', scale=0.35, width=0.005)
axes['quiver'].set_title('In-plane velocity arrows')
axes['quiver'].set_xticks([]); axes['quiver'].set_yticks([])

nonzero = hr_speed[hr_speed > 0.02].cpu()
axes['hist'].hist(nonzero.numpy(), bins=30, color='steelblue')
axes['hist'].set_title('Speed distribution')
axes['hist'].set_xlabel('normalized speed')
axes['hist'].set_ylabel('voxels')
plt.show()

## 2b. High-Resolution Interactive Wall Preview

This preview uses a denser synthetic ground-truth grid than training. It is only for visualization, so it does not increase training cost. The translucent wall is an isosurface of velocity speed, and the cones show sampled 3D velocity vectors.


In [ ]:
preview_hr_size = 64  # try 96 on A100 if interaction stays smooth
hires_preview = SyntheticFlowDataset(samples=1, hr_size=preview_hr_size, noise_std=config['noise_std'], seed=123)
_, hr_hires = hires_preview[0]
fig3d = make_interactive_3d_flow_figure(
    hr_hires,
    wall_threshold=0.03,
    vector_threshold=0.12,
    vector_step=4,
    arrow_scale=0.14,
)
fig3d.show()


## 3. Datasets And Model

In [ ]:
train_dataset = SyntheticFlowDataset(
    samples=config['train_samples'],
    hr_size=config['hr_size'],
    scale=config['scale'],
    noise_std=config['noise_std'],
    seed=10,
)
val_dataset = SyntheticFlowDataset(
    samples=config['val_samples'],
    hr_size=config['hr_size'],
    scale=config['scale'],
    noise_std=config['noise_std'],
    seed=10000,
)

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=0, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))

model = FourDFlowNet(
    width=config['width'],
    lr_blocks=config['lr_blocks'],
    hr_blocks=config['hr_blocks'],
).to(device)

params = sum(p.numel() for p in model.parameters())
print(f'parameters: {params:,}')

## 4. Evaluation Helpers

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    totals = defaultdict(float)
    batches = 0
    for lr, hr in loader:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)
        pred = model(lr)
        interp = upsample_lr(lr, hr.shape[-1])

        totals['model_mae'] += (pred - hr).abs().mean().item()
        totals['interp_mae'] += (interp - hr).abs().mean().item()
        totals['model_epe'] += endpoint_error(pred, hr).item()
        totals['interp_epe'] += endpoint_error(interp, hr).item()
        totals['model_peak_err'] += peak_velocity_error(pred, hr).item()
        totals['interp_peak_err'] += peak_velocity_error(interp, hr).item()
        totals['model_flow_err'] += flow_rate_error(pred, hr).item()
        totals['interp_flow_err'] += flow_rate_error(interp, hr).item()
        totals['model_div_l1'] += divergence_l1(pred).item()
        totals['interp_div_l1'] += divergence_l1(interp).item()
        batches += 1
    return {k: v / batches for k, v in totals.items()}


def print_metrics(epoch, train_loss, metrics):
    print(f'epoch={epoch:03d} train_loss={train_loss:.5f}')
    print(
        '  model:  '
        f"mae={metrics['model_mae']:.5f} epe={metrics['model_epe']:.5f} "
        f"peak_err={100 * metrics['model_peak_err']:.2f}% "
        f"flow_err={100 * metrics['model_flow_err']:.2f}% "
        f"div_l1={metrics['model_div_l1']:.5f}"
    )
    print(
        '  interp: '
        f"mae={metrics['interp_mae']:.5f} epe={metrics['interp_epe']:.5f} "
        f"peak_err={100 * metrics['interp_peak_err']:.2f}% "
        f"flow_err={100 * metrics['interp_flow_err']:.2f}% "
        f"div_l1={metrics['interp_div_l1']:.5f}"
    )

## 5. Train

The loss is close to the paper objective: MSE plus a small velocity-gradient loss. Mixed precision is enabled on GPU.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'])
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
history = []

for epoch in range(1, config['epochs'] + 1):
    model.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{config['epochs']}")
    for lr, hr in pbar:
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            pred = model(lr)
            loss = four_d_flow_loss(pred, hr, gradient_weight=config['gradient_weight'])
            if config['divergence_weight'] > 0:
                loss = loss + config['divergence_weight'] * divergence_l1(pred)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    train_loss = running / len(train_loader)
    metrics = evaluate(model, val_loader)
    metrics['train_loss'] = train_loss
    history.append(metrics)
    print_metrics(epoch, train_loss, metrics)

## 6. Plot Learning Curves

In [ ]:
epochs = np.arange(1, len(history) + 1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)

axes[0].plot(epochs, [h['model_mae'] for h in history], label='model')
axes[0].plot(epochs, [h['interp_mae'] for h in history], label='trilinear')
axes[0].set_title('Velocity MAE')
axes[0].legend()

axes[1].plot(epochs, [100 * h['model_peak_err'] for h in history], label='model')
axes[1].plot(epochs, [100 * h['interp_peak_err'] for h in history], label='trilinear')
axes[1].set_title('Peak velocity error (%)')
axes[1].legend()

axes[2].plot(epochs, [100 * h['model_flow_err'] for h in history], label='model')
axes[2].plot(epochs, [100 * h['interp_flow_err'] for h in history], label='trilinear')
axes[2].set_title('Flow-rate error (%)')
axes[2].legend()

plt.show()

## 7. Visualize Reconstruction

In [ ]:
@torch.no_grad()
def visualize_reconstruction(sample_index=0):
    lr, hr = val_dataset[sample_index]
    lr_b = lr.unsqueeze(0).to(device)
    hr_b = hr.unsqueeze(0).to(device)
    pred = model(lr_b).squeeze(0).cpu()
    interp = upsample_lr(lr_b, hr.shape[-1]).squeeze(0).cpu()
    hr = hr.cpu()

    hr_speed = torch.linalg.vector_norm(hr, dim=0)
    pred_speed = torch.linalg.vector_norm(pred, dim=0)
    interp_speed = torch.linalg.vector_norm(interp, dim=0)
    pred_err = torch.linalg.vector_norm(pred - hr, dim=0)
    interp_err = torch.linalg.vector_norm(interp - hr, dim=0)
    z = hr.shape[1] // 2
    vmax = hr_speed.max().item()

    panels = [
        ('Ground truth', hr_speed[z], 'viridis', 0, vmax),
        ('Trilinear', interp_speed[z], 'viridis', 0, vmax),
        ('4DFlowNet-mini', pred_speed[z], 'viridis', 0, vmax),
        ('Interp error', interp_err[z], 'magma', 0, max(interp_err.max().item(), pred_err.max().item())),
        ('Model error', pred_err[z], 'magma', 0, max(interp_err.max().item(), pred_err.max().item())),
    ]
    fig, axes = plt.subplots(1, len(panels), figsize=(15, 3), constrained_layout=True)
    for ax, (title, image, cmap, vmin, vmax_i) in zip(axes, panels):
        im = ax.imshow(image, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax_i)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

visualize_reconstruction(0)

## 8. Save Checkpoint And Figures

In [ ]:
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
checkpoint = out_dir / '4dflownet_colab.pt'
torch.save({'model': model.state_dict(), 'config': config, 'history': history}, checkpoint)
print('saved', checkpoint.resolve())

## 9. Next Experiments

Try these after the first run:

- Increase `train_samples` to `2048` and `epochs` to `30`.
- Compare `gradient_weight = 0` vs `1e-3`.
- Try `divergence_weight = 1e-4` as a physics-inspired extra term.
- Train on low noise and evaluate on high noise.
- Create a held-out generator mode, e.g. stronger stenosis or stronger branch flow.